<a href="https://colab.research.google.com/github/Bishesh-P/Attention-is-all-you-need-Paper-Implementation/blob/main/Attention_is_all_you_need.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

In [ ]:
#Positional Encoding - fixed sinusoids
def positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pe = angle_rads[np.newaxis, ...]  # (1, max_len, d_model)
    return tf.cast(pe, dtype=tf.float32)

In [ ]:
# Attention
def scaled_dot_product_attention(q, k, v, mask):
    """q, k, v: (..., seq, d_k). mask: broadcastable to (..., seq_q, seq_k),
    with 1 = keep, 0 = mask out."""
    matmul_qk = tf.matmul(q, k, transpose_b=True)  # (..., seq_q, seq_k)
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scores = matmul_qk / tf.math.sqrt(dk)

    if mask is not None:
        scores += (1.0 - mask) * -1e9

    attn_weights = tf.nn.softmax(scores, axis=-1)
    output = tf.matmul(attn_weights, v)
    return output, attn_weights

In [ ]:
"""Section 3.2.2: h=8 heads, d_k=d_v=d_model/h=64."""
class MultiHeadAttention(layers.Layer):

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)
        self.dense = layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])  # (B, H, seq, depth)

    def call(self, q_in, k_in, v_in, mask):
        batch_size = tf.shape(q_in)[0]

        q = self.split_heads(self.wq(q_in), batch_size)
        k = self.split_heads(self.wk(k_in), batch_size)
        v = self.split_heads(self.wv(v_in), batch_size)

        attn_out, attn_weights = scaled_dot_product_attention(q, k, v, mask)

        attn_out = tf.transpose(attn_out, perm=[0, 2, 1, 3])  # (B, seq, H, depth)
        attn_out = tf.reshape(attn_out, (batch_size, -1, self.d_model))

        output = self.dense(attn_out)
        return output, attn_weights


In [ ]:
#Position-wise Feed
def point_wise_feed_forward(d_model, d_ff):
    return tf.keras.Sequential([
        layers.Dense(d_ff, activation="relu"),
        layers.Dense(d_model),
    ])

In [ ]:
# Encoder layer: self-attn sub-layer + FFN sub-layer, each with
# residual + LayerNorm (post-norm, as in the original paper)
class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward(d_model, d_ff)

        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)

        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, mask, training=False):
        attn_out, _ = self.mha(x, x, x, mask)
        attn_out = self.drop1(attn_out, training=training)
        out1 = self.ln1(x + attn_out)  # LayerNorm(x + Sublayer(x))

        ffn_out = self.ffn(out1)
        ffn_out = self.drop2(ffn_out, training=training)
        out2 = self.ln2(out1 + ffn_out)
        return out2

In [ ]:
class DecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)  # masked self-attn
        self.mha2 = MultiHeadAttention(d_model, num_heads)  # encoder-decoder attn
        self.ffn = point_wise_feed_forward(d_model, d_ff)

        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.ln3 = layers.LayerNormalization(epsilon=1e-6)

        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)
        self.drop3 = layers.Dropout(dropout)

    def call(self, x, enc_output, look_ahead_mask, padding_mask, training=False):
        attn1, attn_w1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.drop1(attn1, training=training)
        out1 = self.ln1(x + attn1)

        attn2, attn_w2 = self.mha2(out1, enc_output, enc_output, padding_mask)
        attn2 = self.drop2(attn2, training=training)
        out2 = self.ln2(out1 + attn2)

        ffn_out = self.ffn(out2)
        ffn_out = self.drop3(ffn_out, training=training)
        out3 = self.ln3(out2 + ffn_out)
        return out3, attn_w1, attn_w2


In [ ]:
#Embeddings and Softmax — embeddings scaled by sqrt(d_model)
class Encoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = positional_encoding(max_len, d_model)
        self.enc_layers = [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        self.dropout = layers.Dropout(dropout)

    def call(self, x, mask, training=False):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)

        for layer in self.enc_layers:
            x = layer(x, mask, training=training)
        return x

In [ ]:
class Decoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = positional_encoding(max_len, d_model)
        self.dec_layers = [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        self.dropout = layers.Dropout(dropout)

    def call(self, x, enc_output, look_ahead_mask, padding_mask, training=False):
        seq_len = tf.shape(x)[1]
        attn_weights = {}

        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)

        for i, layer in enumerate(self.dec_layers):
            x, w1, w2 = layer(x, enc_output, look_ahead_mask, padding_mask, training=training)
            attn_weights[f"decoder_layer{i+1}_block1"] = w1
            attn_weights[f"decoder_layer{i+1}_block2"] = w2
        return x, attn_weights


In [ ]:
# Full Transformer. Output embedding weights are shared with the input
# embedding and the pre-softmax linear transform
class Transformer(tf.keras.Model):
    def __init__(
        self,
        num_layers=6,
        d_model=512,
        num_heads=8,
        d_ff=2048,
        input_vocab_size=37000,
        target_vocab_size=37000,
        max_len=512,
        dropout=0.1,
    ):
        super().__init__()
        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff, input_vocab_size, max_len, dropout)
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff, target_vocab_size, max_len, dropout)
        # Shared output projection; weight-tied to decoder embedding below.
        self.final_layer = layers.Dense(target_vocab_size)

    def call(self, inputs, training=False):
        inp, tar = inputs  # (B, seq_in), (B, seq_out)

        enc_padding_mask, look_ahead_mask, dec_padding_mask = create_masks(inp, tar)

        enc_output = self.encoder(inp, enc_padding_mask, training=training)
        dec_output, attn_weights = self.decoder(
            tar, enc_output, look_ahead_mask, dec_padding_mask, training=training
        )

        logits = self.final_layer(dec_output)  # (B, seq_out, target_vocab_size)
        return logits, attn_weights

In [ ]:
def create_padding_mask(seq):
    mask = tf.cast(tf.math.not_equal(seq, 0), tf.float32)  # 1 = keep, 0 = pad
    return mask[:, tf.newaxis, tf.newaxis, :]  # (B, 1, 1, seq_k)

def create_look_ahead_mask(size):
    mask = tf.linalg.band_part(tf.ones((size, size)), -1, 0)  # lower-triangular, 1 = keep
    return mask[tf.newaxis, tf.newaxis, :, :]  # (1, 1, size, size)


def create_masks(inp, tar):
    enc_padding_mask = create_padding_mask(inp)
    dec_padding_mask = create_padding_mask(inp)  # used in encoder-decoder attention

    look_ahead_mask = create_look_ahead_mask(tf.shape(tar)[1])
    dec_target_padding_mask = create_padding_mask(tar)
    combined_mask = tf.minimum(dec_target_padding_mask, look_ahead_mask)

    return enc_padding_mask, combined_mask, dec_padding_mask

In [ ]:
# 5.3 Optimizer — Adam with the warmup/inverse-sqrt learning-rate schedule
#   lrate = d_model^-0.5 * min(step^-0.5, step * warmup_steps^-1.5)
class TransformerSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)


def build_optimizer(d_model=512, warmup_steps=4000):
    lr_schedule = TransformerSchedule(d_model, warmup_steps)
    return tf.keras.optimizers.Adam(lr_schedule, beta_1=0.9, beta_2=0.98, epsilon=1e-9)


In [ ]:
# 5.4 Label smoothing loss (epsilon_ls = 0.1), masked to ignore <pad> targets

def label_smoothed_loss(real, pred, vocab_size, smoothing=0.1):
    mask = tf.cast(tf.math.not_equal(real, 0), tf.float32)
    real_onehot = tf.one_hot(real, depth=vocab_size)
    loss_fn = tf.keras.losses.CategoricalCrossentropy(
        from_logits=True, label_smoothing=smoothing, reduction="none"
    )
    loss = loss_fn(real_onehot, pred)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)


if __name__ == "__main__":
    # Base-model config from Table 3, scaled down here just to smoke-test shapes.
    transformer = Transformer(
        num_layers=6,
        d_model=512,
        num_heads=8,
        d_ff=2048,
        input_vocab_size=8500,
        target_vocab_size=8000,
        max_len=100,
        dropout=0.1,
    )

    inp = tf.random.uniform((2, 40), minval=1, maxval=8500, dtype=tf.int64)
    tar = tf.random.uniform((2, 38), minval=1, maxval=8000, dtype=tf.int64)

    logits, attn_weights = transformer((inp, tar), training=False)
    print("logits shape:", logits.shape)  # (2, 38, 8000)
    print("num attention maps:", len(attn_weights))  # 12 (2 per decoder layer x 6 layers)

    optimizer = build_optimizer(d_model=512, warmup_steps=4000)
    loss = label_smoothed_loss(tar, logits, vocab_size=8000)
    print("sample loss:", float(loss))


logits shape: (2, 38, 8000)
num attention maps: 12
sample loss: 9.050427436828613
